# Processing images and text with VLMs 

This notebook demonstrates how to utilize the `HuggingFaceTB/SmolVLM-Instruct` 4bit-quantized model for various multimodal tasks such as:
- Visual Question Answering (VQA): Answering questions based on image content.
- Text Recognition (OCR): Extracting and interpreting text in images.
- Video Understanding: Describing videos through sequential frame analysis.

By structuring prompts effectively, you can leverage the model for many applications, such as scene understanding, document analysis, and dynamic visual reasoning.

In [ ]:
# Install the requirements in Google Colab
# !pip install -q transformers datasets trl huggingface_hub bitsandbytes accelerate

# Authenticate to Hugging Face
# from huggingface_hub import notebook_login
# notebook_login()

from huggingface_hub import login
login()


/home/ai-makina/.pyenv/versions/smol-course-2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import torch, PIL
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from transformers.image_utils import load_image

device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)

# Updated quantization config to avoid compilation issues
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model_name = "HuggingFaceTB/SmolVLM-Instruct"

try:
    # Use the new model class to avoid deprecation warning
    model = AutoModelForImageTextToText.from_pretrained(
        model_name,
        quantization_config=quantization_config,
        torch_dtype=torch.bfloat16,
        device_map="auto"  # Let transformers handle device placement
    )
    print(f"✅ Model loaded successfully on {device}")
except Exception as e:
    print(f"❌ Error loading quantized model: {e}")
    print("🔄 Trying without quantization...")
    # Fallback: load without quantization if there are issues
    model = AutoModelForImageTextToText.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        device_map="auto"
    )
    print(f"✅ Model loaded without quantization on {device}")

processor = AutoProcessor.from_pretrained("HuggingFaceTB/SmolVLM-Instruct")

print(processor.image_processor.size)

/tmp/tmpgjb319nj/main.c:5:10: fatal error: Python.h: No such file or directory
    5 | #include <Python.h>
      |          ^~~~~~~~~~
compilation terminated.


❌ Error loading quantized model: Command '['/usr/bin/gcc', '/tmp/tmpgjb319nj/main.c', '-O3', '-shared', '-fPIC', '-Wno-psabi', '-o', '/tmp/tmpgjb319nj/cuda_utils.cpython-310-x86_64-linux-gnu.so', '-lcuda', '-L/home/ai-makina/.pyenv/versions/smol-course-2/lib/python3.10/site-packages/triton/backends/nvidia/lib', '-L/lib/x86_64-linux-gnu', '-L/lib/i386-linux-gnu', '-I/home/ai-makina/.pyenv/versions/smol-course-2/lib/python3.10/site-packages/triton/backends/nvidia/include', '-I/tmp/tmpgjb319nj', '-I/usr/include/python3.10']' returned non-zero exit status 1.
🔄 Trying without quantization...
✅ Model loaded without quantization on cuda
✅ Model loaded without quantization on cuda
{'longest_edge': 1536}
{'longest_edge': 1536}


## Processing Images

Let's start with generating captions and answering questions about an image. We'll also explore processing multiple images.
### 1. Single Image Captioning

In [4]:
from IPython.display import Image, display

image_url1 = "https://cdn.pixabay.com/photo/2024/11/20/09/14/christmas-9210799_1280.jpg"
display(Image(url=image_url1))

image_url2 = "https://cdn.pixabay.com/photo/2024/11/23/08/18/christmas-9218404_1280.jpg"
display(Image(url=image_url2))

In [5]:
# Load  one image
image1 = load_image(image_url1)

# Create input messages
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": "Can you describe the image?"}
        ]
    },
]

# Prepare inputs
prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(text=prompt, images=[image1], return_tensors="pt")
inputs = inputs.to(device)

# Generate outputs
generated_ids = model.generate(**inputs, max_new_tokens=500)
generated_texts = processor.batch_decode(
    generated_ids,
    skip_special_tokens=True,
)

print(generated_texts)


['User:<image>Can you describe the image?\nAssistant: The image depicts a scene in a forest, likely taken during the fall season due to the presence of fallen leaves. The primary focus of the image is on two young children, both wearing Santa hats, who are walking away from the camera. The children are dressed in warm winter clothing, including jackets and pants, which suggests it is a cold day.\n\nThe background of the image is dominated by a dense forest, with a variety of trees, including deciduous trees with yellow and brown leaves, and some evergreen trees. The trees are tall and have thick trunks, indicating that the forest is mature. The ground is covered with a layer of fallen leaves, which are predominantly brown and yellow, indicating that it is autumn.\n\nThe children are holding hands, and their bodies are facing away from the camera, suggesting they are walking towards a specific destination. The image captures a moment of interaction between the two children, possibly a p

### 2. Comparing Multiple Images
The model can process and compare multiple images. Let's determine the common theme between two images.

In [6]:

# Load images
image2 = load_image(image_url2)

# Create input messages
messages = [
    # {
    #     "role": "user",
    #     "content": [
    #         {"type": "image"},
    #         {"type": "image"},
    #         {"type": "text", "text": "Can you describe the two images?"}
    #     ]
    # },
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "image"},
            {"type": "text", "text": "What event do they both represent?"}
        ]
    },
]

# Prepare inputs
prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(text=prompt, images=[image1, image2], return_tensors="pt")
inputs = inputs.to(device)

# Generate outputs
generated_ids = model.generate(**inputs, max_new_tokens=500)
generated_texts = processor.batch_decode(
    generated_ids,
    skip_special_tokens=True,
)

print(generated_texts)

['User:<image>What event do they both represent?\nAssistant: Christmas.']


### 🔠 Text Recognition (OCR)
VLM can also recognize and interpret text in images, making it suitable for tasks like document analysis.
You could try experimenting on images with denser text.

In [7]:
document_image_url = "https://cdn.pixabay.com/photo/2020/11/30/19/23/christmas-5792015_960_720.png"
display(Image(url=document_image_url))

# Load the document image
document_image = load_image(document_image_url)

# Create input message for analysis
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": "What is written?"}
        ]
    }
]

# Prepare inputs
prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(text=prompt, images=[document_image], return_tensors="pt")
inputs = inputs.to(device)

# Generate outputs
generated_ids = model.generate(**inputs, max_new_tokens=500)
generated_texts = processor.batch_decode(
    generated_ids,
    skip_special_tokens=True,
)

print(generated_texts)

['User:<image>What is written?\nAssistant: Merry Christmas and a Happy New Year.']


## Processing videos

Visual-Language Models (VLMs) can process videos indirectly by extracting keyframes and reasoning over them in temporal order. While VLMs lack the temporal modeling capabilities of dedicated video models, they can still:
- Describe actions or events by analyzing sampled frames sequentially.
- Answer questions about videos based on representative keyframes.
- Summarize video content by combining textual descriptions of multiple frames.

Let experiment on an example:

<video width="640" height="360" controls>
  <source src="https://cdn.pixabay.com/video/2023/10/28/186794-879050032_large.mp4" type="video/mp4">
  Your browser does not support the video tag.
</video>

In [8]:
# !pip install opencv-python

In [9]:
from IPython.display import Video
import cv2
import numpy as np

def extract_frames(video_path, max_frames=50, target_size=None):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"Could not open video: {video_path}")
    
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_indices = np.linspace(0, total_frames - 1, max_frames, dtype=int)

    frames = []
    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            frame = PIL.Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            if target_size:
                frames.append(resize_and_crop(frame, target_size))
            else:
                frames.append(frame)
    cap.release()
    return frames

def resize_and_crop(image, target_size):
    width, height = image.size
    scale = target_size / min(width, height)
    image = image.resize((int(width * scale), int(height * scale)), PIL.Image.Resampling.LANCZOS)
    left = (image.width - target_size) // 2
    top = (image.height - target_size) // 2
    return image.crop((left, top, left + target_size, top + target_size))

# Video link
video_link = "https://cdn.pixabay.com/video/2023/10/28/186794-879050032_large.mp4"

In [10]:
question = "Describe what the woman is doing."

def generate_response(model, processor, frames, question):

    image_tokens = [{"type": "image"} for _ in frames]
    messages = [
        {
            "role": "user",
            "content": [{"type": "text", "text": "Following are the frames of a video in temporal order."}, *image_tokens, {"type": "text", "text": question}]
        }
    ]
    inputs = processor(
        text=processor.apply_chat_template(messages, add_generation_prompt=True),
        images=frames,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs, max_new_tokens=100, num_beams=5, temperature=0.7, do_sample=True, use_cache=True
    )
    return processor.decode(outputs[0], skip_special_tokens=True)


# Extract frames from the video
frames = extract_frames(video_link, max_frames=15, target_size=384)

processor.image_processor.size = (384, 384)
processor.image_processor.do_resize = False
# Generate response
response = generate_response(model, processor, frames, question)

# Display the result
# print("Question:", question)
print("Response:", response)

OutOfMemoryError: CUDA out of memory. Tried to allocate 2.38 GiB. GPU 0 has a total capacity of 11.56 GiB of which 1.46 GiB is free. Process 386393 has 90.00 MiB memory in use. Including non-PyTorch memory, this process has 10.00 GiB memory in use. Of the allocated memory 8.90 GiB is allocated by PyTorch, and 1014.51 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## 💐 You're Done!

This notebook demonstrated how to use a Visual-Language Model (VLM) such as formating prompts for multimodal tasks. By following the steps outlined here, you can experiment with VLMs and their applications.

### Next Steps to Explore:
- Experiment with more use cases of VLMs.
- Collaborate with a colleague by reviewing their pull requests (PRs).
- Contribute to improving this course material by opening an issue or submitting a PR to introduce new use cases, examples, or concepts.

Happy exploring! 🌟